# Reto de Laboratorio: Preparando la Matriz Tensorial de la Liga ⚽🤖


Contexto de Negocio:
El equipo de analítica quedó fascinado con el Almacén de Datos (OLAP) que construyeron la semana pasada. Ahora, el departamento de Inteligencia Artificial quiere dar el siguiente paso: entrenar una Red Neuronal para predecir cuántos goles anotará un jugador la próxima temporada.El Científico de Datos les acaba de enviar la tabla resultante del cubo OLAP, pero se detuvo por un error de Scikit-Learn: el algoritmo no sabe leer posiciones como "Forward" ni jerarquías como "Oro".Su misión como Ingenieros de Machine Learning es aplicar Feature Engineering para traducir esta tabla humana a una Matriz $\mathbf{X}$ puramente matemática, sin destruir la lógica vectorial.

In [22]:
import pandas as pd
import numpy as np

# 1. Ingesta de datos desde el repositorio crudo
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/players_data-2025_2026.csv"
df_scouting = pd.read_csv(url)

# 2. SIMULACIÓN: El equipo de analistas agregó una columna de jerarquía de talento
# (Evaluación cualitativa del jugador)
np.random.seed(42)
df_scouting['scout_tier'] = np.random.choice(['Bronce', 'Plata', 'Oro', 'Diamante'], size=len(df_scouting))
display(df_scouting)
# Filtramos las columnas que usaremos para el modelo predictivo
columnas_ml = ['Player', 'Squad', 'Age', 'Pos', 'scout_tier', 'MP', 'Gls'] # Taban mal estas variables, no se encontraban las del ejemplo en el dataset lmao
matriz_base = df_scouting[columnas_ml].copy()

print("--- DATOS BASE PARA EL MODELO (Legibles para humanos) ---")
display(matriz_base.head())

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,CrdR_stats_misc,2CrdY,Fls,Fld,Off,Crs,Int,TklW,OG,scout_tier
0,1,Brenden Aaronson,us USA,"MF,FW",Leeds United,eng Premier League,24.0,2000.0,37,30,...,0,0,20,51,5,37,17,27,0,Oro
1,2,Jerome Abbey,eng ENG,MF,Wolves,eng Premier League,15.0,2009.0,1,0,...,0,0,0,0,1,0,1,0,0,Diamante
2,3,Zach Abbott,eng ENG,DF,Nottingham Forest,eng Premier League,19.0,2006.0,3,2,...,0,0,3,1,0,3,2,4,0,Bronce
3,4,Jones El-Abdellaoui,ma MAR,"MF,FW",Celta Vigo,es La Liga,19.0,2006.0,22,4,...,0,0,7,3,2,30,3,3,0,Oro
4,5,Himad Abdelli,dz ALG,MF,Marseille,fr Ligue 1,25.0,1999.0,8,1,...,0,0,3,0,0,1,2,1,0,Oro
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2834,2835,Martín Zubimendi,es ESP,MF,Arsenal,eng Premier League,26.0,1999.0,38,34,...,0,0,33,15,1,13,39,39,0,Oro
2835,2836,Martin Ødegaard,no NOR,MF,Arsenal,eng Premier League,26.0,1998.0,24,16,...,0,0,9,9,2,22,7,9,0,Oro
2836,2837,Luka Đurić,rs SRB,MF,Hoffenheim,de Bundesliga,22.0,2003.0,1,0,...,0,0,0,0,0,0,0,0,0,Plata
2837,2838,Milan Đurić,ba BIH,FW,Cremonese,it Serie A,35.0,1990.0,11,3,...,1,0,4,8,2,2,0,0,0,Bronce


--- DATOS BASE PARA EL MODELO (Legibles para humanos) ---


,Player,Squad,Age,Pos,scout_tier,MP,Gls
0,Brenden Aaronson,Leeds United,24.0,"MF,FW",Oro,37,4
1,Jerome Abbey,Wolves,15.0,MF,Diamante,1,0
2,Zach Abbott,Nottingham Forest,19.0,DF,Bronce,3,0
3,Jones El-Abdellaoui,Celta Vigo,19.0,"MF,FW",Oro,22,2
4,Himad Abdelli,Marseille,25.0,MF,Oro,8,0


### FASE 1: El Traductor Jerárquico (Label Encoding)

Observa la columna `scout_tier`. Contiene valores de texto: 'Bronce', 'Plata', 'Oro', 'Diamante'. Esta variable tiene un **orden natural** claro (Diamante es superior a Bronce).

**Tu tarea:**
Crea un diccionario de mapeo que asigne números enteros respetando la jerarquía natural del talento (iniciando en 1). Luego, aplica este mapeo a la base de datos para crear una nueva columna llamada `scout_encoded`.

In [23]:
encoder = {"Bronce" : 1, "Plata": 2, "Oro": 3, "Diamante": 4}
matriz_base['scout_encoded'] = matriz_base["scout_tier"].map(encoder) 

print("Resultado del Label Encoding:")
display(matriz_base[['Player', 'scout_tier', 'scout_encoded']].head())

Resultado del Label Encoding:


,Player,scout_tier,scout_encoded
0,Brenden Aaronson,Oro,3
1,Jerome Abbey,Diamante,4
2,Zach Abbott,Bronce,1
3,Jones El-Abdellaoui,Oro,3
4,Himad Abdelli,Oro,3


### FASE 2: El Traductor Nominal (One-Hot Encoding)

Ahora observa la columna `position`. Contiene valores como 'Forward' (Delantero), 'Midfielder' (Mediocampista), 'Defender' (Defensa), etc.
Esta variable es **Nominal**. Si le asignamos números (1 al Delantero y 3 al Defensa), la Red Neuronal creerá falsamente que un Defensa "vale el triple" que un Delantero.

**Tu tarea:**
Utiliza la función de Pandas correspondiente para destruir la columna `position` y expandirla en múltiples columnas booleanas (0 y 1). Guarda el resultado en una nueva variable llamada `matriz_X`.

In [24]:
# CÓDIGO ESQUELETO PARA EL ALUMNO:
matriz_X = pd.get_dummies(matriz_base["Pos"], dtype=int)

print("Resultado del One-Hot Encoding:")
display(matriz_X.head())

Resultado del One-Hot Encoding:


,DF,"DF,FW","DF,MF",FW,"FW,MF",GK,MF,"MF,DF","MF,FW"
0,0,0,0,0,0,0,0,0,1
1,0,0,0,0,0,0,1,0,0
2,1,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,1,0,0


### FASE 3: Purgando la Basura Humana

La Inteligencia Artificial está casi lista para devorar los datos. Sin embargo, si le enviamos la matriz ahora mismo, el algoritmo fallará porque todavía hay texto puro estorbando.

**Tu tarea:**
Elimina de `matriz_X` todas las columnas que ya no le sirven al algoritmo:
1. Las columnas de texto original que ya transformaste (como `scout_tier`).
2. Las columnas descriptivas/operativas que no aportan valor matemático universal (`player_name`, `team_name`).

In [26]:
matriz_X_completa = pd.concat([matriz_X, matriz_base], axis=1)
matriz_X_completa = matriz_X_completa.drop(columns=["Player", "Squad", "Pos", "scout_tier"])
display(matriz_X_completa)

print("--- MATRIZ X DEFINITIVA (Lista para Scikit-Learn / PyTorch) ---")
display(matriz_X_completa.head())
print("\nTipos de datos finales (Todos deben ser numéricos):")
print(matriz_X_completa.dtypes)

,DF,"DF,FW","DF,MF",FW,"FW,MF",GK,MF,"MF,DF","MF,FW",Age,MP,Gls,scout_encoded
0,0,0,0,0,0,0,0,0,1,24.0,37,4,3
1,0,0,0,0,0,0,1,0,0,15.0,1,0,4
2,1,0,0,0,0,0,0,0,0,19.0,3,0,1
3,0,0,0,0,0,0,0,0,1,19.0,22,2,3
4,0,0,0,0,0,0,1,0,0,25.0,8,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2834,0,0,0,0,0,0,1,0,0,26.0,38,5,3
2835,0,0,0,0,0,0,1,0,0,26.0,24,1,3
2836,0,0,0,0,0,0,1,0,0,22.0,1,0,2
2837,0,0,0,1,0,0,0,0,0,35.0,11,0,1


--- MATRIZ X DEFINITIVA (Lista para Scikit-Learn / PyTorch) ---


,DF,"DF,FW","DF,MF",FW,"FW,MF",GK,MF,"MF,DF","MF,FW",Age,MP,Gls,scout_encoded
0,0,0,0,0,0,0,0,0,1,24.0,37,4,3
1,0,0,0,0,0,0,1,0,0,15.0,1,0,4
2,1,0,0,0,0,0,0,0,0,19.0,3,0,1
3,0,0,0,0,0,0,0,0,1,19.0,22,2,3
4,0,0,0,0,0,0,1,0,0,25.0,8,0,3



Tipos de datos finales (Todos deben ser numéricos):
DF                 int64
DF,FW              int64
DF,MF              int64
FW                 int64
FW,MF              int64
GK                 int64
MF                 int64
MF,DF              int64
MF,FW              int64
Age              float64
MP                 int64
Gls                int64
scout_encoded      int64
dtype: object


### 🧠 FASE 4: Defensa Arquitectónica (Responde aquí mismo)

Imagina que el CTO de la empresa revisa tu código y te hace dos preguntas críticas. Responde con argumentos de ingeniería:

**Pregunta 1:** *"Noté que borraste la columna `team_name` en lugar de aplicarle One-Hot Encoding. En nuestra base de datos mundial de la FIFA tenemos 4,000 equipos diferentes. Si le hubieras aplicado One-Hot Encoding a la columna de equipos, ¿qué le habría pasado a la estructura matemática de tu Matriz X y a la memoria RAM de nuestro servidor?"*
> **Tu Respuesta:** A la estructura matemática nos hubiera dado el problema de la dimensionalidad debido a que hubiera creado 4k columnas diferentes y la RAM hubiera desaparecido solo para procesar todos esos datos.

**Pregunta 2:** *"¿Por qué usamos Label Encoding (1, 2, 3, 4) para la evaluación del scout (`scout_tier`), pero no usamos esa misma técnica matemática para la posición en el campo (`position`)?"*
> **Tu Respuesta:** Porque label encoding lleva el peso de la informacion jerarquica, no podemos aplicar ese tipo para una columna en que los atributos no tengan jerarquía como la posición